# Scoring an Embedding Without Looking at It

**Topic:** Neighbourhood preservation &nbsp;·&nbsp; **Difficulty:** Medium

"The t-SNE plot looks good" is not a measurement. Trustworthiness is: it counts
how many points that are close in the embedding were *not* close in the original
space, penalized by how far away they really were.

Implement it, and use it to compare PCA against t-SNE on a curved manifold where
the visual answer and the numeric answer agree, which is not always the case.

## The data (created in the next cell)

- `X` `(400, 3)`: points on a Swiss-roll-like curved 2D manifold.
- `k = 12` neighbours for the metric.

## Your task

With **sklearn and NumPy**:

1. Implement trustworthiness for an embedding `Z`:
   `T = 1 - (2 / (n k (2n - 3k - 1))) * sum_i sum_{j in U_i} (r_ij - k)`
   where `U_i` is the set of points among `Z`'s `k` nearest neighbours of `i`
   that are **not** among `X`'s `k` nearest neighbours of `i`, and `r_ij` is the
   rank of `j` among `i`'s neighbours in the **original** space (rank 1 =
   nearest).
2. Embed with `PCA(n_components=2)` and with
   `TSNE(n_components=2, perplexity=30, random_state=0, init="pca")`.
3. Set `answers` to a dict with `"trust_pca"`, `"trust_tsne"`, and
   `"trust_random"`, the score of a random 2D embedding drawn with
   `np.random.default_rng(0).normal(0, 1, (400, 2))`.

## Evaluation

Compared to the reference: relative tolerance 1e-6 for the PCA and random
scores, and 1e-3 for t-SNE, whose optimisation is not bit-for-bit reproducible
across scikit-learn versions. The evaluation cell
checks all scores lie in `[0, 1]`, that t-SNE beats PCA, and that both beat
random.

In [ ]:
import numpy as np

rng = np.random.default_rng(3011)
n = 400
t = rng.uniform(1.5 * np.pi, 4.0 * np.pi, n)
h = rng.uniform(0, 12, n)
X = np.column_stack([t * np.cos(t), h, t * np.sin(t)])
k = 12
print(f"X {X.shape}: a Swiss roll, intrinsically 2D but curved in 3D")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("trust_pca", [], 0.9439433158584535, -0.1580519949047552, 1e-06),
    ("trust_tsne", [], 0.998255242463958, -0.16714587607633166, 1e-03),
    ("trust_random", [], 0.5063805155089558, -0.08478734825756226, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if all(_k in answers for _k in ("trust_pca", "trust_tsne", "trust_random")):
    _range = all(0 <= answers[k2] <= 1 for k2 in ("trust_pca", "trust_tsne", "trust_random"))
    _order = (answers["trust_tsne"] > answers["trust_pca"] > answers["trust_random"])
    print(f"  {'scores in [0,1]':<24} {'ok' if _range else 'wrong'}")
    print(f"  {'tsne > pca > random':<24} {'ok' if _order else 'wrong'}  "
          f"({answers['trust_tsne']:.3f} > {answers['trust_pca']:.3f} > "
          f"{answers['trust_random']:.3f})")
    _ok = _ok and _range and _order

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

The metric asks a specific question: of the points that *look* like neighbours in
the embedding, how badly were they misplaced in the original space? A point that
was rank 13 costs almost nothing, while a point that was rank 300 costs a lot. That
asymmetry is deliberate: trustworthiness penalizes false neighbours, while its
partner metric continuity penalizes missed ones, and an embedding can be good at
one and bad at the other.

```python
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

n = len(X)

def ranks_and_neighbors(A, k):
    D = np.sqrt(((A[:, None, :] - A[None, :, :]) ** 2).sum(axis=2))
    np.fill_diagonal(D, np.inf)
    order = np.argsort(D, axis=1)
    rank = np.empty_like(order)
    rows = np.arange(A.shape[0])[:, None]
    rank[rows, order] = np.arange(1, A.shape[0] + 1)[None, :]
    return rank, order[:, :k]

rank_X, nn_X = ranks_and_neighbors(X, k)

def trustworthiness(Z):
    _, nn_Z = ranks_and_neighbors(Z, k)
    total = 0.0
    for i in range(n):
        orig = set(nn_X[i].tolist())
        for j in nn_Z[i]:
            if int(j) not in orig:
                total += rank_X[i, j] - k
    norm = 2.0 / (n * k * (2 * n - 3 * k - 1))
    return float(1.0 - norm * total)

Zp = PCA(n_components=2).fit_transform(X)
Zt = TSNE(n_components=2, perplexity=30, random_state=0, init="pca").fit_transform(X)
Zr = np.random.default_rng(0).normal(0, 1, (n, 2))

answers = {
    "trust_pca": trustworthiness(Zp),
    "trust_tsne": trustworthiness(Zt),
    "trust_random": trustworthiness(Zr),
}
```